# Compute Fashion-CLIP Embeddings
Recompute image and text embeddings using `patrickjohncyh/fashion-clip` — a CLIP ViT-B/32 model fine-tuned on 800k fashion product images with rich attribute labels.

**Why Fashion-CLIP over standard CLIP?**
- Standard CLIP: trained on 400M general image-text pairs
- Fashion-CLIP: additionally fine-tuned on fashion-specific data → tighter clusters for same-product items

**Output files:**
- `fashion_clip_image_embeddings.pt` — `{str(itemId): tensor(512)}` for all train + phase1 items
- `fashion_clip_text_embeddings.pt` — `{str(itemId): tensor(512)}` for all train + phase1 items

After running this notebook, swap the paths in `train_and_predict.ipynb` and retrain.

In [1]:
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np
import os
from PIL import Image
from tqdm.auto import tqdm
from transformers import CLIPProcessor, CLIPModel

# ── Device ─────────────────────────────────────────────────────────────────────
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(f"Device: {device}")

# ── Paths ──────────────────────────────────────────────────────────────────────
DATA_FOLDER    = "../data/"
IMAGES_FOLDER  = "../../images/"   # DO NOT LIST THIS FOLDER — open per-item only
IMAGE_OUT_PATH = "fashion_clip_image_embeddings.pt"
TEXT_OUT_PATH  = "fashion_clip_text_embeddings.pt"
MODEL_ID       = "patrickjohncyh/fashion-clip"

print(f"Will save image embeddings → {IMAGE_OUT_PATH}")
print(f"Will save text  embeddings → {TEXT_OUT_PATH}")


Device: mps
Will save image embeddings → fashion_clip_image_embeddings.pt
Will save text  embeddings → fashion_clip_text_embeddings.pt


In [2]:
# ── Load Fashion-CLIP model & processor ────────────────────────────────────────
print(f"Loading {MODEL_ID} ...")
processor = CLIPProcessor.from_pretrained(MODEL_ID)
model     = CLIPModel.from_pretrained(MODEL_ID).to(device)
model.eval()

EMB_DIM = model.config.projection_dim   # 512 for ViT-B/32
print(f"  Embedding dim : {EMB_DIM}")
print(f"  Vision model  : {model.config.vision_config.model_type}")
print("Model loaded!")


Loading patrickjohncyh/fashion-clip ...


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/568 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: patrickjohncyh/fashion-clip
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Embedding dim : 512
  Vision model  : clip_vision_model
Model loaded!


In [3]:
# ── Load item metadata for train + phase1 ─────────────────────────────────────
df_train  = pd.read_csv(os.path.join(DATA_FOLDER, "items_train.csv"))
df_phase1 = pd.read_csv(os.path.join(DATA_FOLDER, "items_phase_1.csv"))

# Combine — deduplicate by itemId so we encode each item once
df_all = pd.concat([df_train, df_phase1], ignore_index=True)
df_all = df_all.drop_duplicates(subset="itemId").reset_index(drop=True)
df_all["itemId_str"] = df_all["itemId"].astype(str)

df_all["title"]       = df_all["title"].fillna("")
df_all["description"] = df_all["description"].fillna("")
df_all["text"]        = (df_all["title"] + " " + df_all["description"]).str.strip()

print(f"Train items  : {len(df_train):,}")
print(f"Phase1 items : {len(df_phase1):,}")
print(f"Total unique : {len(df_all):,}")


Train items  : 928,234
Phase1 items : 199,835
Total unique : 1,128,069


In [9]:
# ── Text embeddings using multilingual CLIP ────────────────────────────────────
# sentence-transformers/clip-ViT-B-32-multilingual-v1 is the multilingual
# version of the SAME ViT-B/32 used for image embeddings → spaces are aligned.
# Outputs 512d vectors, handles CZ, SK, HU, RO, PL, EN, etc.

from sentence_transformers import SentenceTransformer

MULTILINGUAL_TEXT_OUT_PATH = "text_embeddings.pt"

if os.path.exists(MULTILINGUAL_TEXT_OUT_PATH):
    text_embeddings = torch.load(MULTILINGUAL_TEXT_OUT_PATH, map_location="cpu", weights_only=False)
    print(f"Resuming — already have {len(text_embeddings):,} text embeddings")
else:
    text_embeddings = {}

print("Loading sentence-transformers/clip-ViT-B-32-multilingual-v1...")
text_model = SentenceTransformer("sentence-transformers/clip-ViT-B-32-multilingual-v1", device=str(device))
print(f"  Embedding dim: {text_model.get_sentence_embedding_dimension()}")

texts_map      = df_all.set_index("itemId_str")["text"].to_dict()
remaining_text = [iid for iid in item_ids if iid not in text_embeddings]
print(f"Items to encode: {len(remaining_text):,}")

TEXT_BATCH_SIZE = 512  # SentenceTransformer handles batching internally but we chunk for checkpointing

for batch_start in tqdm(range(0, len(remaining_text), TEXT_BATCH_SIZE), desc="Text embeddings"):
    batch_ids   = remaining_text[batch_start : batch_start + TEXT_BATCH_SIZE]
    batch_texts = [texts_map.get(iid, "") for iid in batch_ids]

    # encode() returns numpy array (B, 512), already L2-normalised by default
    with torch.no_grad():
        feats = text_model.encode(
            batch_texts,
            batch_size=TEXT_BATCH_SIZE,
            convert_to_tensor=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        ).cpu()  # (B, 512)

    for iid, feat in zip(batch_ids, feats):
        text_embeddings[iid] = feat

    # Checkpoint every ~50k items
    if (batch_start // TEXT_BATCH_SIZE) % 100 == 0:
        torch.save(text_embeddings, MULTILINGUAL_TEXT_OUT_PATH)

torch.save(text_embeddings, MULTILINGUAL_TEXT_OUT_PATH)
print(f"\nSaved {len(text_embeddings):,} multilingual text embeddings → {MULTILINGUAL_TEXT_OUT_PATH}")


Loading sentence-transformers/clip-ViT-B-32-multilingual-v1...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

  Embedding dim: 512
Items to encode: 1,128,069
Items to encode: 1,128,069


Text embeddings:   0%|          | 0/2204 [00:00<?, ?it/s]


Saved 1,128,069 multilingual text embeddings → text_embeddings.pt
